In [11]:
import yfinance as yf
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, timedelta,date
import requests
import json
from polygon import RESTClient
from datetime import date as date_type, timedelta
import pandas as pd
import re
import numpy as np


## Historical Price of Underlying Asset

In [12]:
symbol = "AAPL"
ticker = yf.Ticker(symbol)
history = ticker.history(start="2023-12-01")
df = pd.DataFrame(history)
df.reset_index(inplace=True)
df['Date'] = df['Date'].dt.date
df['Return'] = df['Close'].pct_change()
df['RV'] = df['Return'].rolling(window=20).std() * np.sqrt(252)
df.to_csv(f"DataSet/underlying.csv", index=False)


## Trade every first Tuesday, get the list of trade dates

In [13]:
start_date = date(2024, 3, 1)
end_date = date(2026, 1, 31)

trade_dates = []
trade_dates_str = []
current = date(start_date.year, start_date.month, 1)

while current <= end_date:
    days_to_tuesday = (1 - current.weekday()) % 7
    first_tuesday = current.replace(day=1) + timedelta(days=days_to_tuesday)
    if start_date <= first_tuesday <= end_date:
        trade_dates.append(first_tuesday)
        trade_dates_str.append(first_tuesday.strftime("%Y-%m-%d"))
    if current.month == 12:
        current = date(current.year + 1, 1, 1)
    else:
        current = date(current.year, current.month + 1, 1)

# for d, ds in zip(trade_dates, trade_dates_str):
#     print(repr(d), ds)

## Get the underlying Price at each trading dates

## 1-month Treasury yield as risk-free rate over time (Polygon)
Polygon does not provide 1-month OIS; we use **1-month U.S. Treasury yield** from Polygon's Fed API as the risk-free rate for each date.

In [14]:
# Fetch 1-month Treasury yield over time (Polygon) as risk-free rate
# Polygon does not offer 1-month OIS; we use 1-month Treasury yield (closest short-term risk-free proxy).
api_key = "XN1r3nHQ1Rb3SsbqdwDI72dE35vJkCzP"
client = RESTClient(api_key)
from_date = min(trade_dates).strftime("%Y-%m-%d")
to_date = date.today().strftime("%Y-%m-%d")
# yields_raw: list of TreasuryYield(date='YYYY-MM-DD', yield_1_month=5.48, yield_3_month=..., ...); yield_1_month in %.
yields_raw = list(client.list_treasury_yields(date_gte=from_date, date_lte=to_date, limit=50000, sort="date", order="asc"))
r_fallback = 0.0365
# print(yields_raw[0].date,yields_raw[0].yield_1_month)
if not yields_raw:
    r_by_date = pd.Series(r_fallback, index=pd.date_range(from_date, to_date, freq="D"))
    print("No Treasury yields from Polygon; using fallback rate for all dates.")
else:
    # Loop over yields_raw: record date and yield_1_month for each i; yield in % -> decimal r
    dates_list = []
    rates_list = []
    for i in range(len(yields_raw)):
        d = yields_raw[i].date
        y = yields_raw[i].yield_1_month
        dates_list.append(pd.to_datetime(d))
        rates_list.append((float(y) / 100.0) if y is not None else np.nan)
    r_by_date = pd.Series(rates_list, index=dates_list)
    r_by_date = r_by_date.sort_index()
    # Missing calendar days: use rate from previous day (ffill)
    full_range = pd.date_range(from_date, to_date, freq="D")
    r_by_date = r_by_date.reindex(full_range).ffill().fillna(r_fallback)
    r_by_date = r_by_date.astype(np.float64)
    print(f"Loaded 1-month Treasury yields from {r_by_date.index.min()} to {r_by_date.index.max()} (n={len(r_by_date)})")

Loaded 1-month Treasury yields from 2024-03-05 00:00:00 to 2026-02-25 00:00:00 (n=723)


In [15]:
s0_list = []
underlying_df = pd.read_csv("DataSet/underlying.csv", parse_dates=['Date'])
underlying_df.set_index('Date', inplace=True)

for date_str in trade_dates_str:
    date_pd = pd.to_datetime(date_str)
    close_price = underlying_df.loc[date_pd]['Close']
    s0_list.append(close_price)
# print(s0_list)

## Get option date. Always buy Option expired in the second friday of next month

In [16]:
def atm_option(symbol, s0, date):
    api_key = "XN1r3nHQ1Rb3SsbqdwDI72dE35vJkCzP"
    client = RESTClient(api_key)

    # date is guaranteed to be a datetime.date
    date_obj = date

    date_str = date_obj.strftime("%Y-%m-%d")
    # print(f"Using spot price s0: {s0} on date: {date_str}")

    # Calculate second Friday of next month
    if date_obj.month == 12:
        next_month_first = date_obj.replace(year=date_obj.year + 1, month=1, day=1)
    else:
        next_month_first = date_obj.replace(month=date_obj.month + 1, day=1)
    
    # weekday(): 0=Mon, 1=Tue, ..., 4=Fri
    days_to_friday = (4 - next_month_first.weekday()) % 7
    first_friday = next_month_first + timedelta(days=days_to_friday)
    second_friday = first_friday + timedelta(weeks=1)
    exp_date_str = second_friday.strftime("%Y-%m-%d")
    # print(f"Target expiration (2nd Friday next month): {exp_date_str}")

    # Get options contracts as of the given date (limit=1000 for efficiency, even with unlimited calls)
    contracts = list(client.list_options_contracts(underlying_ticker=symbol, as_of=date_str, limit=1000))
    if not contracts:
        print(f"No options contracts found as of {date_str}.")
        return None, None

    # Filter calls and puts for exact expiration
    calls = [c for c in contracts if c.expiration_date == exp_date_str and c.contract_type == 'call']
    puts = [c for c in contracts if c.expiration_date == exp_date_str and c.contract_type == 'put']

    if not calls or not puts:
        print(f"No calls or puts found for expiration {exp_date_str} as of {date_str}.")
        return None, None

    # Find ATM call and put (closest strike to s0)
    atm_call = min(calls, key=lambda c: abs(c.strike_price - s0))
    atm_put = min(puts, key=lambda c: abs(c.strike_price - s0))

    # print(f"ATM Call: {atm_call.ticker} (strike: {atm_call.strike_price})")
    # print(f"ATM Put: {atm_put.ticker} (strike: {atm_put.strike_price})")
    if (atm_call.strike_price != atm_put.strike_price):
        print("Warning, Strike price of call and put are not the same")
    # else:
        # print(abs(atm_call.strike_price - s0))

    return atm_call.ticker, atm_put.ticker

## Download the data, and calculate the implied volitality

In [ ]:
# Black-Scholes formula for calls and puts
def bs_call_price(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

def bs_put_price(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r*T) * norm.cdf(-d2) - S * norm.cdf(-d1)

IV_FALLBACK = 1.0

def implied_vol_call(market_price, S, K, T, r):
    def obj_func(sigma):
        return bs_call_price(S, K, T, r, sigma) - market_price
    try:
        return brentq(obj_func, 1e-6, 5)
    except ValueError:
        return IV_FALLBACK

def implied_vol_put(market_price, S, K, T, r):
    def obj_func(sigma):
        return bs_put_price(S, K, T, r, sigma) - market_price
    try:
        return brentq(obj_func, 1e-6, 5)
    except ValueError:
        return IV_FALLBACK

def download_hist_option(symbol, start_date, r, s0, underlying_df=None):
    """
    Downloads historical daily data for the given option ticker starting from the given date to today.
    
    Args:
        symbol (str): Option ticker, e.g., 'O:AAPL240209C00185000'
        start_date (date): Start date for historical data.
        r (float or pd.Series): Risk-free rate. If Series (index=date), use actual rate per date (e.g. 1-month Treasury).
        s0 (float): Underlying spot price (fixed for all historical dates; for accuracy, consider passing historical prices).
    
    Returns:
        pd.DataFrame: Historical data with columns: timestamp, open, high, low, close, volume, ttm, imp_vol
    """
    api_key = ""
    client = RESTClient(api_key)
    
    # Define date range: start from given date to today
    from_str = start_date.strftime("%Y-%m-%d")
    to_str = date.today().strftime("%Y-%m-%d")
    
    # Fetch daily aggregates for the option
    aggs = client.get_aggs(
        ticker=symbol,
        multiplier=1,
        timespan="day",
        from_=from_str,
        to=to_str,
        limit=50000  # High limit to cover all days
    )
    
    if not aggs:
        print(f"No historical data found for {symbol} from {from_str}.")
        return pd.DataFrame()
    
    # Convert to DataFrame
    df = pd.DataFrame([
        {
            "timestamp": pd.to_datetime(agg.timestamp, unit="ms").date(),
            "open": agg.open,
            "high": agg.high,
            "low": agg.low,
            "close": agg.close,
            "volume": agg.volume,
            "imp_vol": getattr(agg, 'implied_volatility', None) or getattr(agg, 'iv', None)
        }
        for agg in aggs
    ])
    
    # Sort by date
    df = df.sort_values("timestamp").reset_index(drop=True)
    
    # Extract expiry date, option type, and strike from symbol
    symbol_part = symbol.split(':')[1]
    match = re.search(r'^[A-Z]+(\d{6})([CP])(\d{8})$', symbol_part)
    if not match:
        raise ValueError(f"Could not parse option symbol: {symbol}. Expected format like 'AAPL240209C00185000'.")
    
    expiry_str = match.group(1)
    option_type = match.group(2)
    strike_str = match.group(3)
    
    # Normalize expiry to 8 digits
    if len(expiry_str) == 6:
        expiry_str = '20' + expiry_str  # Assumes 21st century
    
    expiry = pd.to_datetime(expiry_str, format='%Y%m%d').date()
    
    # Strike price
    K = int(strike_str) / 1000.0
    
    # Calculate time to maturity (ttm) in years for each row
    df['ttm'] = df['timestamp'].apply(lambda x: max((expiry - x).days / 365.25, 1e-9))
    # Risk-free rate: scalar or per-date Series (e.g. 1-month Treasury from Polygon)
    if isinstance(r, pd.Series):
        df['r'] = r.reindex(pd.to_datetime(df['timestamp'])).ffill().bfill().fillna(0.0365).values
    else:
        df['r'] = r
    df['k'] = K
    
    # Use underlying close (S) per date when available; else fixed s0
    if underlying_df is not None:
        df['date'] = pd.to_datetime(df['timestamp']).dt.date
        spot_df = underlying_df[['Close']].reset_index()
        spot_df.columns = ['date', 'S']
        spot_df['date'] = pd.to_datetime(spot_df['date']).dt.date
        df = df.merge(spot_df, on='date', how='left')
        df['S'] = df['S'].fillna(s0)
        df = df.drop(columns=['date'])
    else:
        df['S'] = s0
    
    # Use API implied_volatility when present and valid; else compute from close (Black-Scholes)
    df['imp_vol'] = pd.to_numeric(df['imp_vol'], errors='coerce')
    need_iv = df['imp_vol'].isna() | ~np.isfinite(df['imp_vol'])
    if need_iv.any():
        if option_type == 'C':
            df.loc[need_iv, 'imp_vol'] = df.loc[need_iv].apply(lambda row: implied_vol_call(row['close'], row['S'], K, row['ttm'], row['r']), axis=1)
        else:
            df.loc[need_iv, 'imp_vol'] = df.loc[need_iv].apply(lambda row: implied_vol_put(row['close'], row['S'], K, row['ttm'], row['r']), axis=1)
    df['imp_vol'] = df['imp_vol'].astype(float)
    df = df.drop(columns=['S'])
    
    # print(f"Fetched {len(df)} rows for {symbol} from {from_str} to {to_str}.")
    
    csv_name = f"DataSet/{symbol.replace(':', '_')}.csv"
    df.to_csv(csv_name, index=False)
    
    return df

## Build All Data

In [18]:
call_list = []
put_list = []
# last_trade_date = trade_dates[0]
for idx in range(len(s0_list)):
    date = trade_dates[idx]
    date_str = trade_dates_str[idx]
    s0 = s0_list[idx]
    atm_call, atm_put = atm_option(symbol, s0, date)
    call_list.append(atm_call.replace(':', '_'))
    put_list.append(atm_put.replace(':', '_'))
    download_hist_option(atm_call, date, r_by_date, s0, underlying_df)
    download_hist_option(atm_put, date, r_by_date, s0, underlying_df)
    # last_trade_date = date

## Store Data

In [19]:
import pickle
with open('DataSet/call_list.pkl', 'wb') as f:
    pickle.dump(call_list, f)
with open('DataSet/put_list.pkl', 'wb') as f:
    pickle.dump(put_list, f)
with open('DataSet/dates.pkl', 'wb') as f:
    pickle.dump(trade_dates, f)
with open('DataSet/date_strs.pkl', 'wb') as f:
    pickle.dump(trade_dates_str, f)

## Add imp_vol to underlying.csv
For each date, store (call_iv + put_iv) / 2 from the option pair that has data on that date.

In [20]:
# Build date -> (call_iv + put_iv)/2 from the option pair that has that date
date_to_iv = {}
for idx in range(len(call_list)):
    call_path = f"DataSet/{call_list[idx]}.csv"
    put_path = f"DataSet/{put_list[idx]}.csv"
    call_df = pd.read_csv(call_path)
    put_df = pd.read_csv(put_path)
    call_df["date"] = pd.to_datetime(call_df["timestamp"]).dt.date
    put_df["date"] = pd.to_datetime(put_df["timestamp"]).dt.date
    call_iv = call_df.set_index("date")["imp_vol"]
    put_iv = put_df.set_index("date")["imp_vol"]
    for d in call_iv.index:
        if d in put_iv.index:
            c_iv, p_iv = call_iv.loc[d], put_iv.loc[d]
            if pd.notna(c_iv) and pd.notna(p_iv):
                date_to_iv[d] = (float(c_iv) + float(p_iv)) / 2

underlying_with_iv = pd.read_csv("DataSet/underlying.csv", parse_dates=["Date"])
underlying_with_iv["Date_only"] = pd.to_datetime(underlying_with_iv["Date"]).dt.date
underlying_with_iv["imp_vol"] = underlying_with_iv["Date_only"].map(date_to_iv)
underlying_with_iv["r"] = r_by_date.reindex(pd.to_datetime(underlying_with_iv["Date_only"])).ffill().bfill().fillna(r_fallback).values
underlying_with_iv = underlying_with_iv.drop(columns=["Date_only"])
underlying_with_iv["VRP"] = underlying_with_iv["imp_vol"] - underlying_with_iv["RV"]
underlying_with_iv["VRP_std"] = underlying_with_iv["VRP"].rolling(window=20, min_periods=1).std()
underlying_with_iv["VRP_mean"] = underlying_with_iv["VRP"].rolling(window=20, min_periods=1).mean()
underlying_with_iv.to_csv("DataSet/underlying.csv", index=False)
print("Added imp_vol column to underlying.csv")

Added imp_vol column to underlying.csv
